# Phase 2 — Official test run and final comparison (15)

Issue #71. Every model the project tuned is trained once more with its final settings and scored **once** on the official FD001 test set. This is the project's main deliverable: engineered-feature regression against sequence learning, on engines no model or setting has seen.

**Terms**

- **Official test set:** `test_FD001.txt`, 100 engines whose records stop some time before failure. `RUL_FD001.txt` gives each engine's true remaining useful life (RUL) at its last recorded cycle, so each engine is scored once, at that cycle, against the uncapped true RUL.
- **Model freeze:** the commit whose code and configs produce every number below. After this run, no model or config may change. The notebook stops if `src/`, `configs/` or the saved tuning results differ from the freeze commit.
- **Cap:** training labels are cut at 125 cycles (#55). Every model is also trained uncapped, and both are shown, as the protocol asks.
- **Refit:** training on all 100 training engines. Networks train for a fixed number of epochs: the median best epoch of the 3-fold cross-validation that tuned them, at seed 42 (`results/metrics/fd001_cv_<model>.json`). The epoch count is part of the frozen setting, so the five seeds change only the random initialization and batch order.
- **Seeds:** 42 to 46. Networks move by up to about 2 cycles between seeds (#68), so the networks are reported as mean ± standard deviation over the five seeds. The ML baselines are scored once with the #61 option (`scripts/run_ml_baselines.py`), at seed 42.
- **Stages:** late (true RUL ≤ 50), mid (50 < RUL ≤ 100) and early (RUL > 100), as in every Phase 2 notebook.

**Models**

| Model | Issue | Config |
|---|---|---|
| LSTM | #58 | `configs/lstm.yaml` |
| Temporal CNN | #59 | `configs/tcn.yaml` |
| Transformer | #60 | `configs/transformer.yaml` |
| Hybrid A: gated fusion | #64, #68 | `configs/hybrid_gated.yaml` |
| Hybrid B: XGBoost on cross-fitted encoder embeddings | #65, #68 | `configs/hybrid_two_stage.yaml` |
| Random Forest, XGBoost, Elastic Net on raw and engineered inputs | #61 | `results/metrics/fd001_ml_capped_cv.json` |

Re-running this notebook reloads the saved test predictions. It does not retrain or rescore anything once `results/metrics/fd001_final_test_predictions.csv` exists.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "scripts"))

import run_ml_baselines as mlb
from rul.data.load import load_subset
from rul.evaluation.metrics import evaluate
from rul.evaluation.official import METRICS_DIR, predict_test_cycles, save_results, score_last_cycle
from rul.training.config import apply_overrides, load_config, run_cv, run_refit
from rul.training.runs import save_cv

FREEZE_COMMIT = "fff9e06aad09ba9547388f2db9c9b72e023b400c"
SUBSET = "FD001"
SEEDS = [42, 43, 44, 45, 46]
MODELS = {
    "lstm": "LSTM",
    "tcn": "Temporal CNN",
    "transformer": "Transformer",
    "hybrid_gated": "Hybrid A (gated)",
    "hybrid_two_stage": "Hybrid B (two-stage)",
}
TARGETS = {"cap125": [], "uncapped": ["target.cap=null"]}
STAGES = ("late", "mid", "early")

DATA_DIR = ROOT / "data" / "raw" / "CMAPSSData"
OUT_METRICS = METRICS_DIR
FIG_DIR = ROOT / "results" / "figures"
MODELS_DIR = ROOT / "results" / "models" / "final"   # gitignored checkpoints and the resumable cache


def git(*args):
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout


# The freeze: the code, the configs and the tuning results (as they were at the freeze) that fix every setting.
FROZEN = ["src", "configs", "scripts/run_ml_baselines.py", "results/metrics/fd001_ml_capped_cv.json",
          *[f"results/metrics/fd001_cv_{m}{s}.json" for m in MODELS for s in ("", "_uncapped")
            if subprocess.run(["git", "cat-file", "-e", f"{FREEZE_COMMIT}:results/metrics/fd001_cv_{m}{s}.json"],
                              cwd=ROOT, capture_output=True).returncode == 0]]
changed = git("diff", "--name-only", FREEZE_COMMIT, "--", *FROZEN).split()
assert not changed, f"changed since the freeze commit: {changed}"
print("freeze commit", FREEZE_COMMIT[:7], "| HEAD", git("rev-parse", "--short", "HEAD").strip())
print("frozen paths unchanged:", ", ".join(FROZEN[:4]), f"+ {len(FROZEN) - 4} tuning results")

In [ ]:
train, test, rul_test = load_subset(SUBSET, data_dir=DATA_DIR)
DRY_EPOCHS = None
print(f"training engines {train['unit'].nunique()}, test engines {test['unit'].nunique()}, "
      f"true test RUL {rul_test.min()} to {rul_test.max()} cycles")

## Refit epochs

Each network refits for the median best epoch of the cross-validation that tuned it (seed 42, early stopping on validation RMSE). The saved cross-validation must have been run with exactly the config used here. Hybrid A had no uncapped cross-validation yet, so it runs once here on the training engines; it does not read the test set. Hybrid B fits its own way (cross-fitted encoders with early stopping, then XGBoost), so it needs no epoch count.

In [ ]:
def config_for(model, target, seed=None):
    overrides = TARGETS[target] + ([f"training.seed={seed}"] if seed is not None else [])
    return apply_overrides(load_config(ROOT / "configs" / f"{model}.yaml"), overrides)


def run_name(model, target):
    return model if target == "cap125" else f"{model}_uncapped"


epochs, cv_scores = {}, {}
for model in MODELS:
    for target in TARGETS:
        if model == "hybrid_two_stage":
            epochs[model, target] = None
            continue
        config = config_for(model, target)
        path = METRICS_DIR / f"fd001_cv_{run_name(model, target)}.json"
        if not path.exists():
            cv = run_cv(config, train)
            save_cv(path.stem, cv, meta={"name": run_name(model, target), "subset": SUBSET,
                                         "seed": 42, "config": config})
        saved = json.loads(path.read_text())
        assert saved["meta"]["config"] == config, f"{path.name} was run with a different config"
        epochs[model, target] = saved["median_best_epoch"]
        cv_scores[model, target] = saved["summary"]["selection_rmse"]

pd.DataFrame([
    {"model": MODELS[m], "target": t, "refit epochs": epochs[m, t] if epochs[m, t] else "own",
     "CV selection RMSE (seed 42)": f"{cv_scores[m, t]['mean']:.2f} ± {cv_scores[m, t]['std']:.2f}"
     if (m, t) in cv_scores else "—"}
    for m in MODELS for t in TARGETS
])

## Train and score once

For each model, target and seed: refit on all training engines, save the checkpoint (gitignored), predict every test cycle and keep each engine's last cycle. Finished runs are appended to a cache, so an interrupted run resumes where it stopped instead of training again.

In [ ]:
FINAL = OUT_METRICS / f"{SUBSET.lower()}_final_test_predictions.csv"
CACHE = MODELS_DIR / "test_predictions_cache.csv"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
KEYS = ["model", "target", "seed"]

if FINAL.exists():
    predictions = pd.read_csv(FINAL)
    print(f"loaded {FINAL.name}: nothing is retrained or rescored")
else:
    done = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame(columns=KEYS)
    finished = set(map(tuple, done[KEYS].astype(str).to_numpy()))
    for model in MODELS:
        for target in TARGETS:
            for seed in SEEDS:
                if (model, target, str(seed)) in finished:
                    continue
                config = config_for(model, target, seed)
                n_epochs = DRY_EPOCHS or epochs[model, target]
                trained, seconds = run_refit(config, train, n_epochs)
                trained.save(MODELS_DIR / f"{SUBSET.lower()}_{run_name(model, target)}_seed{seed}.pt",
                             meta={"config": config, "epochs": n_epochs, "seconds": seconds,
                                   "freeze_commit": FREEZE_COMMIT})
                last, _ = score_last_cycle(predict_test_cycles(trained, test, rul_test))
                part = last[["unit", "rul", "prediction"]].assign(
                    model=model, target=target, seed=seed, epochs=n_epochs, seconds=round(seconds, 1))
                part.to_csv(CACHE, mode="a", header=not CACHE.exists(), index=False)
                print(f"{model:17s} {target:8s} seed {seed}: {seconds:5.0f} s, "
                      f"RMSE {np.sqrt(np.mean((part.prediction - part.rul) ** 2)):.2f}")
    predictions = (pd.read_csv(CACHE)
                   .sort_values(["model", "target", "seed", "unit"]).reset_index(drop=True))
    runs = {m: {t: {str(s): evaluate(g["rul"], g["prediction"])
                    for s, g in predictions[(predictions.model == m) & (predictions.target == t)]
                    .groupby("seed")}
                for t in TARGETS} for m in MODELS}
    save_results(FINAL.name.removesuffix("_predictions.csv"), {
        "subset": SUBSET, "issue": 71, "freeze_commit": FREEZE_COMMIT, "seeds": SEEDS,
        "scored_at": "last recorded cycle of each test engine, against the uncapped true RUL",
        "refit_epochs": {f"{m}/{t}": epochs[m, t] for m in MODELS for t in TARGETS},
        "runs": runs,
    }, predictions[["model", "target", "seed", "unit", "rul", "prediction", "epochs", "seconds"]],
        out_dir=OUT_METRICS)

print(predictions.groupby(KEYS).size().unstack("seed").to_string())

## ML baselines, scored once

The #61 script's `--score-test` option refits each chosen setting on all training engines and scores the test set at each engine's last cycle. The chosen settings are read from the saved cross-validation, so the grid is not searched again.

In [ ]:
ML_TEST = OUT_METRICS / f"{SUBSET.lower()}_ml_capped_test.json"
if not ML_TEST.exists():
    ml_cv = json.loads((METRICS_DIR / f"{SUBSET.lower()}_ml_capped_cv.json").read_text())
    chosen = pd.DataFrame([{"variant": c["variant"], "cap": c["cap"], "model": c["model"],
                            "params": json.dumps(c["params"])} for c in ml_cv["chosen"]])
    mlb.score_test(chosen, mlb.parse_args(["--subset", SUBSET, "--data-dir", str(DATA_DIR),
                                           "--out-dir", str(OUT_METRICS)]))
ml_test = json.loads(ML_TEST.read_text())["models"]
print(json.dumps({v: {c: {m: round(s["rmse"], 2) for m, s in ms.items()} for c, ms in cs.items()}
                  for v, cs in ml_test.items()}, indent=1))

## Results table

Networks: mean ± standard deviation over the five seeds. ML baselines: one run. Stage columns are RMSE over the test engines whose true RUL falls in that stage at their last cycle. The NASA score sums an exponential penalty over engines and punishes late predictions (more life promised than is left) harder than early ones; lower is better.

In [ ]:
ML_NAMES = {"random_forest": "Random Forest", "xgboost": "XGBoost", "elastic_net": "Elastic Net"}
rows = []
for (model, target, seed), g in predictions.groupby(KEYS):
    scores = evaluate(g["rul"], g["prediction"])
    rows.append({"family": "sequence" if model in ("lstm", "tcn", "transformer") else "hybrid",
                 "model": MODELS[model], "target": target, "seed": seed,
                 **{k: scores[k] for k in ("rmse", "mae", "nasa_score")},
                 **{f"{s}_rmse": scores["by_stage"][s]["rmse"] for s in STAGES}})
for variant, caps in ml_test.items():
    for cap, models in caps.items():
        for name, scores in models.items():
            rows.append({"family": f"ML, {variant}", "model": f"{ML_NAMES[name]} ({variant})",
                         "target": "cap125" if cap == "125" else "uncapped", "seed": 42,
                         **{k: scores[k] for k in ("rmse", "mae", "nasa_score")},
                         **{f"{s}_rmse": scores["by_stage"][s]["rmse"] for s in STAGES}})
per_run = pd.DataFrame(rows)
order = list(MODELS.values()) + [f"{ML_NAMES[n]} ({v})" for v in ("engineered", "raw") for n in ML_NAMES]
metrics = ["rmse", "mae", "nasa_score", *[f"{s}_rmse" for s in STAGES]]
summary = (per_run.groupby(["family", "model", "target"], sort=False)[metrics]
           .agg(["mean", "std"]).reset_index())
summary.columns = [" ".join(c).strip().replace(" mean", "") for c in summary.columns]
summary.insert(3, "seeds", per_run.groupby(["family", "model", "target"], sort=False).size().to_numpy())
summary["model"] = pd.Categorical(summary["model"], order, ordered=True)
summary = summary.sort_values(["target", "model"]).reset_index(drop=True)
summary.round(3).to_csv(OUT_METRICS / f"{SUBSET.lower()}_final_comparison.csv", index=False)


def cell(row, col, digits=2):
    sd = row[f"{col} std"]
    return f"{row[col]:.{digits}f}" + (f" ± {sd:.{digits}f}" if pd.notna(sd) and row["seeds"] > 1 else "")


capped = summary[summary.target == "cap125"].set_index("model")
uncapped = summary[summary.target == "uncapped"].set_index("model")
table = pd.DataFrame({
    "seeds": capped["seeds"],
    "RMSE": [cell(r, "rmse") for _, r in capped.iterrows()],
    "MAE": [cell(r, "mae") for _, r in capped.iterrows()],
    "NASA score": [cell(r, "nasa_score", 0) for _, r in capped.iterrows()],
    **{f"{s} RMSE": [f"{v:.2f}" for v in capped[f"{s}_rmse"]] for s in STAGES},
    "RMSE uncapped": [cell(r, "rmse") for _, r in uncapped.loc[capped.index].iterrows()],
}, index=capped.index)
table

## Figures

Left: test RMSE of every model, trained with the cap (filled) and uncapped (hollow); bars are ± one standard deviation over seeds. Right: RMSE by life stage for the capped models. Below: each test engine's prediction at its last cycle against its true RUL, for the best network and the best ML baseline, engines sorted by true RUL.

In [ ]:
COLORS = {"sequence": "#3b6ea8", "hybrid": "#c0504d", "ML, engineered": "#4f8f4f", "ML, raw": "#9a9a9a"}
STAGE_COLORS = {"late": "#c0504d", "mid": "#e3a33b", "early": "#3b6ea8"}
fig, (ax, bx) = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
labels = list(capped.index)
y = np.arange(len(labels))[::-1]
for target, frame, face in (("cap 125", capped, None), ("uncapped", uncapped.loc[labels], "none")):
    colors = [COLORS[f] for f in frame["family"]]
    ax.errorbar(frame["rmse"], y, xerr=frame["rmse std"], fmt="none", ecolor="#555", capsize=3, lw=1)
    ax.scatter(frame["rmse"], y, s=55, c=colors if face is None else "white",
               edgecolors=colors, linewidths=1.6, zorder=3, label=target)
ax.set_yticks(y, labels)
ax.set_xlabel("Test RMSE (cycles), last cycle of 100 engines")
legend = ax.legend(loc="lower right", bbox_to_anchor=(1, 1.0), ncol=2, frameon=False)
for handle, face in zip(legend.legend_handles, ("#444", "white")):
    handle.set_edgecolor("#444")
    handle.set_facecolor(face)
ax.grid(axis="x", alpha=0.3)
ax.set_title("All models", loc="left")

height = 0.27
n_stage = predictions.query("model == 'lstm' and target == 'cap125' and seed == @SEEDS[0]")["rul"]
counts = {"late": (n_stage <= 50).sum(), "mid": ((n_stage > 50) & (n_stage <= 100)).sum(),
          "early": (n_stage > 100).sum()}
for i, stage in enumerate(STAGES):
    bx.barh(y + (1 - i) * height, capped[f"{stage}_rmse"], height, xerr=capped[f"{stage}_rmse std"],
            color=STAGE_COLORS[stage], label=f"{stage} ({counts[stage]} engines)",
            error_kw={"lw": 0.8, "capsize": 2})
bx.set_xlabel("RMSE by life stage (cycles), cap 125")
bx.legend(loc="lower right", bbox_to_anchor=(1, 1.0), ncol=3, frameon=False, fontsize=8)
bx.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / f"{SUBSET.lower()}_final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
networks = capped[capped["family"].isin(["sequence", "hybrid"])]
best_net = networks["rmse"].idxmin()
best_net_key = next(k for k, v in MODELS.items() if v == best_net)
ml_rows = capped[capped["family"].str.startswith("ML")]
best_ml = ml_rows["rmse"].idxmin()
ml_preds = pd.read_csv(OUT_METRICS / f"{SUBSET.lower()}_ml_capped_test_predictions.csv")
ml_name, ml_variant = best_ml.rsplit(" (", 1)
ml_col = f"{ml_variant.rstrip(')')}_{next(k for k, v in ML_NAMES.items() if v == ml_name)}_cap125"

net = (predictions.query("model == @best_net_key and target == 'cap125'")
       .groupby("unit").agg(rul=("rul", "first"), mean=("prediction", "mean"),
                            low=("prediction", "min"), high=("prediction", "max")))
order_units = net.sort_values("rul").index
net = net.loc[order_units]
ml_line = ml_preds.set_index("unit").loc[order_units, ml_col]

fig, ax = plt.subplots(figsize=(13, 4.6))
pos = np.arange(len(order_units))
ax.plot(pos, net["rul"], color="black", lw=1.5, label="true RUL")
ax.vlines(pos, net["low"], net["high"], color="#c0504d", alpha=0.35, lw=2)
ax.scatter(pos, net["mean"], s=14, color="#c0504d", label=f"{best_net}, mean of {len(SEEDS)} seeds (bar: range)", zorder=3)
ax.scatter(pos, ml_line, s=14, marker="x", color="#4f8f4f", label=f"{best_ml}", zorder=3)
ax.axhline(125, color="#888", ls=":", lw=1)
ax.text(1, 127, "training cap 125", color="#666", fontsize=8)
ax.set_xlabel("Test engines, sorted by true RUL")
ax.set_ylabel("RUL at the last cycle (cycles)")
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / f"{SUBSET.lower()}_final_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## Findings

_Findings are written after the run._